# Notebook 01 — Google Earth Engine desde cero

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/teledeteccion/blob/main/sesiones/sesion-02/colab/01_intro_gee_python.ipynb)

## Teledetección — Maestría en Ingeniería, Universidad del Magdalena
**Sesión 2 · Sábado 18 de julio de 2026**

---

### ¿Qué es Google Earth Engine?

**Google Earth Engine (GEE)** es una plataforma en la nube que tiene almacenados
millones de imágenes satelitales (Sentinel, Landsat, MODIS y más) y te permite
procesarlas **sin descargarlas** — todo corre en los servidores de Google.

Lo que en un computador portátil tardaría horas procesando una imagen de 600 MB,
en GEE tarda segundos.

**Acceso gratuito para investigación y educación** → https://earthengine.google.com

### Lo que aprenderás en este notebook
1. Instalar y autenticar GEE en Google Colab
2. Cargar una colección de imágenes Sentinel-2
3. Filtrar por zona geográfica, fecha y nubosidad
4. Visualizar composiciones de color en un mapa interactivo

---
**Prerrequisito:** Debes tener una cuenta aprobada en GEE.
Si no la tienes aún → https://code.earthengine.google.com (seleccionar Academia/Non-profit)

---

## Paso 1 — Instalar librerías

En Google Colab, `earthengine-api` ya viene instalado. Solo necesitamos instalar
`geemap`, que es la librería que nos permite hacer mapas interactivos dentro del notebook.

In [4]:
# Instalar geemap (solo la primera vez — tarda ~1 minuto)
# El signo ! al inicio indica que es un comando del sistema, no Python
!pip install geemap --quiet

print("✓ Instalación completada")

✓ Instalación completada


---

## Paso 2 — Autenticación e inicialización

La primera vez que uses GEE en Colab debes autenticarte con tu cuenta de Google.
Esto vincula este notebook con tu proyecto de GEE.

**¿Qué va a pasar?**
1. Aparece un link — haz click y abre en una nueva pestaña
2. Selecciona tu cuenta de Google
3. Copia el código de autorización
4. Pégalo en el campo que aparece en esta celda
5. Presiona Enter

In [5]:
import ee
import geemap

# Autenticar (abre ventana del navegador para iniciar sesión con Google)
# Solo necesitas hacer esto una vez por sesión de Colab
ee.Authenticate()

# Inicializar con tu proyecto de GEE
# Reemplaza 'tu-proyecto-gee' con el ID de tu proyecto
# (lo encuentras en https://code.earthengine.google.com → Settings → Projects)
ee.Initialize(project='teledeteccion-miguepoloc')

print("✓ Google Earth Engine inicializado correctamente")
print("✓ geemap versión:", geemap.__version__)

✓ Google Earth Engine inicializado correctamente
✓ geemap versión: 0.38.3


---

## Paso 3 — Conceptos fundamentales de GEE

GEE tiene su propio vocabulario. Antes de escribir código, necesitas entender tres conceptos:

| Concepto GEE | Qué es | Analogía |
|-------------|--------|----------|
| `ee.Geometry` | Una forma geométrica (punto, rectángulo, polígono) | El área que dibujaste en el mapa |
| `ee.Image` | Una imagen satelital con sus bandas | Una foto de satélite de un momento específico |
| `ee.ImageCollection` | Una pila de imágenes ordenadas por fecha | Un álbum de fotos del mismo lugar a lo largo del tiempo |

El flujo típico en GEE es:
1. Definir el **área de estudio** (`ee.Geometry`)
2. Cargar la **colección** de imágenes del satélite que necesitas
3. **Filtrar** por zona, fecha y calidad
4. **Reducir** la colección a una sola imagen (median, mean, mosaic)
5. **Visualizar** en el mapa

In [6]:
# ============================================================
# PASO 3A: DEFINIR EL ÁREA DE ESTUDIO
# ============================================================

# ee.Geometry.Rectangle([longitud_oeste, latitud_sur, longitud_este, latitud_norte])
# Las coordenadas negativas = oeste / sur (Colombia está en el hemisferio norte
# pero al oeste del meridiano 0, por eso la longitud es negativa)

# Zona cacaotera entre Ciénaga y Fundación, Magdalena
zona_cacaotera = ee.Geometry.Rectangle([-74.2, 10.5, -73.8, 11.0])

# Norte del Magdalena (contexto regional más amplio)
norte_magdalena = ee.Geometry.Rectangle([-74.5, 10.2, -73.2, 11.2])

print("Áreas de estudio definidas:")
print("  Zona cacaotera SNSM: Ciénaga–Fundación, Magdalena")
print("  Norte del Magdalena: contexto regional")

Áreas de estudio definidas:
  Zona cacaotera SNSM: Ciénaga–Fundación, Magdalena
  Norte del Magdalena: contexto regional


In [7]:
# ============================================================
# PASO 3B: CARGAR Y FILTRAR LA COLECCIÓN SENTINEL-2
# ============================================================

# ID de la colección en GEE: 'COPERNICUS/S2_SR_HARMONIZED'
# S2_SR = Sentinel-2 Surface Reflectance (Level 2A, ya corregida atmosféricamente)
# HARMONIZED = armonizada entre Sentinel-2A y 2B (calibración consistente)

coleccion_s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    # Filtro 1: Solo imágenes que cubran nuestra zona de estudio
    .filterBounds(zona_cacaotera)
    # Filtro 2: Solo temporada seca 2024 (enero-marzo → menos nubes en la SNSM)
    .filterDate('2024-01-01', '2024-03-31')
    # Filtro 3: Solo imágenes con menos del 15% de nubes
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15))
)

# ¿Cuántas imágenes pasaron los filtros?
n_imagenes = coleccion_s2.size()
print('Imágenes Sentinel-2 disponibles (ene-mar 2024, < 15% nubes):', n_imagenes.getInfo())

Imágenes Sentinel-2 disponibles (ene-mar 2024, < 15% nubes): 46


In [8]:
# ============================================================
# PASO 3C: REDUCIR LA COLECCIÓN A UNA IMAGEN
# ============================================================

# .median() combina todas las imágenes tomando el valor MEDIANO de cada píxel
# Ventaja: elimina nubes residuales (una nube en enero no aparece en marzo)
# Resultado: la imagen más limpia posible del período seleccionado

imagen_mediana = (
    coleccion_s2
    .median()              # reducir colección → una sola imagen
    .clip(zona_cacaotera)  # recortar al área de estudio
)

# Ver qué bandas tiene la imagen
bandas = imagen_mediana.bandNames()
print("Bandas disponibles en la imagen:")
print(bandas.getInfo())

Bandas disponibles en la imagen:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


---

## Paso 4 — Visualizar en mapa interactivo

`geemap.Map()` crea un mapa interactivo dentro del notebook — similar a Google Maps
pero con capas satelitales que tú controlas.

In [9]:
# ============================================================
# VISUALIZACIÓN 1: COLOR NATURAL
# ============================================================

# Parámetros de visualización
# bands: qué bandas van a los canales Rojo, Verde, Azul de la pantalla
# min/max: rango de valores a mapear al rango de colores (0-255)
# Los valores de Sentinel-2 L2A van de 0 a ~10000

params_color_natural = {
    'bands': ['B4', 'B3', 'B2'],   # Rojo, Verde, Azul del espectro
    'min': 0,
    'max': 3000,
    'gamma': 1.4   # aumenta el brillo ligeramente
}

# Crear el mapa
mapa = geemap.Map()
mapa.centerObject(zona_cacaotera, zoom=11)

# Agregar la capa de color natural
mapa.addLayer(
    imagen_mediana,
    params_color_natural,
    'Sentinel-2 — Color Natural (B4-B3-B2)'
)

# Agregar el contorno de la zona de estudio
mapa.addLayer(
    ee.Image().paint(zona_cacaotera, 1, 2),  # pintar solo el borde
    {'palette': ['FF6B00']},
    'Zona Cacaotera'
)

# Mostrar el mapa
mapa

Map(center=[10.749994930545569, -73.99999999999935], controls=(WidgetControl(options=['position', 'transparent…

In [10]:
# ============================================================
# VISUALIZACIÓN 2: FALSO COLOR (VEGETACIÓN = ROJO BRILLANTE)
# ============================================================

# Al asignar el NIR (B8) al canal Rojo de la pantalla:
# → la vegetación aparece ROJA (porque refleja mucho en NIR)
# → el agua aparece casi NEGRA (porque absorbe el NIR)
# → el suelo desnudo aparece en tonos CAFÉ/BEIGE

params_falso_color = {
    'bands': ['B8', 'B4', 'B3'],   # NIR, Rojo, Verde
    'min': 0,
    'max': 4000
}

mapa2 = geemap.Map()
mapa2.centerObject(zona_cacaotera, zoom=11)

# Agregar ambas composiciones para comparar con los controles de capas
mapa2.addLayer(imagen_mediana, params_color_natural, 'Color Natural')
mapa2.addLayer(imagen_mediana, params_falso_color,   'Falso Color NIR')

mapa2

Map(center=[10.749994930545569, -73.99999999999935], controls=(WidgetControl(options=['position', 'transparent…

**Usa los controles de capas** (ícono de capas arriba a la derecha del mapa) para
activar y desactivar las dos composiciones. Observa cómo cambia la apariencia de
la misma zona con diferentes asignaciones de bandas.

---

## Paso 5 — Explorar valores de píxeles individuales

In [11]:
# Extraer los valores de las bandas para puntos específicos
# Esto es el equivalente a hacer click con el Inspector en el Code Editor de GEE

# Definir puntos de muestreo en diferentes coberturas
puntos = {
    'Zona cacaotera (Ciénaga)':     ee.Geometry.Point([-74.00, 10.85]),
    'Bosque SNSM (ladera)':          ee.Geometry.Point([-73.90, 10.92]),
    'Río Magdalena':                  ee.Geometry.Point([-74.18, 10.70]),
    'Zona urbana Fundación':          ee.Geometry.Point([-74.19, 10.52]),
    'Ciénaga Grande (cuerpo de agua)':ee.Geometry.Point([-74.35, 10.78]),
}

bandas_analizar = ['B2', 'B3', 'B4', 'B8', 'B11']
nombres_bandas  = ['B2 (Azul)', 'B3 (Verde)', 'B4 (Rojo)', 'B8 (NIR)', 'B11 (SWIR)']

print(f"{'Cobertura':<35} {'B2':>6} {'B3':>6} {'B4':>6} {'B8 NIR':>8} {'B11 SWIR':>10}")
print("-" * 75)

for nombre, punto in puntos.items():
    valores = imagen_mediana.select(bandas_analizar).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=punto.buffer(30),  # buffer de 30 m = un píxel de Sentinel-2
        scale=10
    ).getInfo()
    
    b2  = int(valores.get('B2',  0) or 0)
    b3  = int(valores.get('B3',  0) or 0)
    b4  = int(valores.get('B4',  0) or 0)
    b8  = int(valores.get('B8',  0) or 0)
    b11 = int(valores.get('B11', 0) or 0)
    
    print(f"{nombre:<35} {b2:>6} {b3:>6} {b4:>6} {b8:>8} {b11:>10}")

print()
print("Observa:")
print("  → Zona cacaotera/Bosque: B8 (NIR) es mucho más alto que B4 (Rojo) → NDVI alto")
print("  → Río/Ciénaga: B8 (NIR) es muy bajo → agua absorbe en NIR → NDVI negativo")
print("  → Zona urbana: B8 y B4 son similares → NDVI cercano a 0")

Cobertura                               B2     B3     B4   B8 NIR   B11 SWIR
---------------------------------------------------------------------------
Zona cacaotera (Ciénaga)               591    719    584     3805       1917
Bosque SNSM (ladera)                   644    813    947     2290       2995
Río Magdalena                          393    775    528     4118       2572
Zona urbana Fundación                 1350   1629   1840     2392       3202
Ciénaga Grande (cuerpo de agua)          0      0      0        0          0

Observa:
  → Zona cacaotera/Bosque: B8 (NIR) es mucho más alto que B4 (Rojo) → NDVI alto
  → Río/Ciénaga: B8 (NIR) es muy bajo → agua absorbe en NIR → NDVI negativo
  → Zona urbana: B8 y B4 son similares → NDVI cercano a 0


---

## Paso 6 — Comparar temporadas: seca vs. húmeda

In [12]:
# Cargar imagen de temporada HÚMEDA (octubre-noviembre 2024)
# Objetivo: ver el efecto de las nubes en el Caribe colombiano

imagen_humeda = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(zona_cacaotera)
    .filterDate('2024-10-01', '2024-11-30')
    # Nota: NO filtramos por nubosidad para ver la diferencia real
    .median()
    .clip(zona_cacaotera)
)

# Contar imágenes disponibles en temporada húmeda (sin filtro de nubes)
n_humeda = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(zona_cacaotera)
    .filterDate('2024-10-01', '2024-11-30')
    .size()
)

n_seca = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(zona_cacaotera)
    .filterDate('2024-01-01', '2024-03-31')
    .size()
)

print("Imágenes disponibles temporada SECA  (ene-mar 2024):", n_seca.getInfo())
print("Imágenes disponibles temporada HÚMEDA (oct-nov 2024):", n_humeda.getInfo())
print()
print("La cantidad de imágenes útiles no es el problema — el problema")
print("es que muchas están cubiertas de nubes en temporada húmeda.")

Imágenes disponibles temporada SECA  (ene-mar 2024): 72
Imágenes disponibles temporada HÚMEDA (oct-nov 2024): 48

La cantidad de imágenes útiles no es el problema — el problema
es que muchas están cubiertas de nubes en temporada húmeda.


In [13]:
# Mapa comparativo: seca vs. húmeda
mapa3 = geemap.Map()
mapa3.centerObject(zona_cacaotera, zoom=11)

mapa3.addLayer(imagen_mediana, params_color_natural, 'Temporada SECA (ene-mar 2024)',  True)
mapa3.addLayer(imagen_humeda,  params_color_natural, 'Temporada HÚMEDA (oct-nov 2024)', False)

mapa3

Map(center=[10.749994930545569, -73.99999999999935], controls=(WidgetControl(options=['position', 'transparent…

**Activa la capa húmeda** y observa cuánto de la imagen está cubierto por nubes.
Este es exactamente el problema que mencionamos en la teoría:
en el Caribe colombiano hay 60–80% de días nublados en temporada húmeda.

Por eso en el Artículo 3 de investigación usamos **Sentinel-1 SAR**,
que penetra las nubes y captura imágenes igual en cualquier condición.

---

## ✅ Resumen — Lo que aprendiste en este notebook

| Concepto | Lo que hiciste |
|----------|---------------|
| `ee.Geometry.Rectangle` | Definir la zona cacaotera de la SNSM |
| `ee.ImageCollection` | Cargar el catálogo Sentinel-2 L2A |
| `.filterBounds()` | Filtrar imágenes que cubren tu zona |
| `.filterDate()` | Solo temporada seca enero–marzo |
| `.filter(lt('CLOUDY_PIXEL_PERCENTAGE', 15))` | Máximo 15% de nubes |
| `.median().clip()` | Obtener imagen limpia del período |
| `geemap.Map()` | Mapa interactivo con capas satelitales |
| `.bandNames()` | Ver las 13 bandas de Sentinel-2 |
| `.reduceRegion()` | Extraer valores de puntos específicos |

---

## ➡️ Siguiente: Notebook 02 — Calcular NDVI sobre la SNSM

Ya sabemos cargar y visualizar imágenes. En el siguiente notebook calcularemos
el NDVI y lo interpretaremos sobre la zona cacaotera.